# Data Lifecycle & Privacy

Two operational primitives the photo app has been missing: deleting things, and serving things that private metadata does not leak out of. Notebook 12 has no `DELETE /photos/{id}` — the gallery can grow but never shrink, and a duplicate you discover today sits in your library forever, hiding in search results. The EXIF extension note in notebook 06 extracts GPS coordinates and stores them as columns on `photos`, but the derivatives we serve back to the client (introduced in PHT:02) keep EXIF in the JPEG bytes by default, because Pillow's `Image.save` writes EXIF unless explicitly told not to. Anyone who downloads a thumbnail gets the photographer's GPS coordinates baked into the JPEG.

[Delete and strip are the two features that make this app safe to share]{.mark}. Delete without cascade leaves orphaned rows and orphaned S3 objects that accumulate forever; stray EXIF silently discloses home addresses, hotel rooms, and travel patterns. This notebook closes both. We build a `DELETE /photos/{id}` that cascades across every table and bucket the photo touches, an audit log that records every mutation with the responsible `device_id`, a soft-delete tombstone with a 30-day grace period so accidental deletes are recoverable, an EXIF stripper that runs on every derivative we generate, and per-owner export/purge endpoints so a family member's photos leave the library cleanly when they ask.

---

## The Delete Cascade

Deleting a photo touches:

1. Postgres: the `photos` row, its `faces` rows (one per detected face), its `summaries` row (if multimodal narration was generated), its `photo_derivatives` rows (PHT:02), its face-cluster membership rows, and its row in any embedding table.
2. S3: the original object in the photos bucket, the three derivative objects (`small`, `medium`, `large`) in the thumbnails bucket, and the per-face thumbnail objects in the same thumbnails bucket.
3. Redis: the presigned-URL cache entries (PHT:03) for any `(bucket, key)` pair we just removed.

If any one of those steps runs but the others do not, we have dangling references: rows pointing at deleted S3 objects, or S3 objects with no row (charged against your storage bill forever). The cascade must run inside a Postgres transaction with a single outermost commit; S3 deletes are best-effort after the commit (S3 deletions are not transactional); Redis invalidation is also after commit.


In [ ]:
from dataclasses import dataclass, field


@dataclass
class PhotoCascadeTargets:
    """All artifacts that must disappear when a photo is deleted."""
    photo_id:           str
    s3_original_key:    str
    s3_original_bucket: str
    derivative_keys:    dict[str, str] = field(default_factory=dict)   # {"small":"thumbnails/..", ...}
    derivative_bucket:  str = ""
    face_thumbnail_keys: list[str] = field(default_factory=list)        # ["thumbnails/faces/{face_id}.jpg", ...]
    summary_id:         str | None = None


def load_cascade_targets(photo_id: str, db) -> PhotoCascadeTargets:
    """Read everything we need to delete before deleting anything. In SQL:

        SELECT p.s3_key, p.sha256,
               (SELECT jsonb_object_agg(size_name, s3_key)
                  FROM photo_derivatives WHERE photo_id = p.photo_id) AS derivs,
               (SELECT array_agg(s3_key) FROM face_thumbnails WHERE photo_id = p.photo_id) AS faces,
               (SELECT summary_id FROM summaries WHERE photo_id = p.photo_id) AS summary_id
        FROM photos p WHERE photo_id = :pid;
    """
    # Stub returns a pre-seeded row so the demo runs.
    return db.get(photo_id, PhotoCascadeTargets(photo_id, "photos/aa/abcd.jpg", "my-photos"))


async def delete_photo_cascade(
    photo_id: str,
    db,                                         # provides load_cascade_targets
    s3_client,
    cache,                                       # Redis (or in-memory stand-in)
    presign_expiry_s: int = 3600,
) -> None:
    """The full cascade, in three ordered phases:
    load (read), commit-and-delete (write), best-effort cleanup (S3+Redis).
    Defined here as the canonical version; the demo cell below seeds
    a tiny DB + cache and runs it.
    """
    targets = load_cascade_targets(photo_id, db)
    if targets.photo_id is None:
        raise KeyError(photo_id)

    db.pop(photo_id, None)   # In prod: cascading via FK ON DELETE

    s3_keys_to_delete = [
        (targets.s3_original_bucket, targets.s3_original_key),
    ]
    for key in targets.derivative_keys.values():
        if key:
            s3_keys_to_delete.append((targets.derivative_bucket, key))
    for key in targets.face_thumbnail_keys:
        if key:
            s3_keys_to_delete.append((targets.derivative_bucket, key))

    for bucket, key in s3_keys_to_delete:
        s3_client.delete_object(Bucket=bucket, Key=key)

    buckets = list({b for b, _ in s3_keys_to_delete})
    keys    = list({k for _, k in s3_keys_to_delete})
    await invalidate_photo_urls(cache, keys, buckets, expiry_s=presign_expiry_s)


### The presign cache invalidate helper

The last line of the cascade relies on an `invalidate_photo_urls` helper
that mirrors the one from PHT:03. We redefine the small in-memory cache and
key+delete helpers inline so this notebook is executable on its own.


In [ ]:
import hashlib
import time
from unittest.mock import MagicMock


class _InMemoryRedis:
    def __init__(self):
        self._store: dict[str, tuple[str, float]] = {}
    async def get(self, k):
        v, exp = self._store.get(k, (None, 0.0))
        return v if (v is not None and exp > time.time()) else None
    async def set(self, k, v, ex):
        self._store[k] = (v, time.time() + ex)
    async def delete(self, k):
        self._store.pop(k, None)


def _cache_key(bucket: str, key: str, expiry_s: int) -> str:
    h = hashlib.sha256(f"{bucket}|{key}|{expiry_s}".encode("utf-8")).hexdigest()[:16]
    return f"presign:get:{h}"


async def invalidate_photo_urls(cache, s3_keys, buckets, expiry_s=3600):
    for b in buckets:
        for key in s3_keys:
            await cache.delete(_cache_key(b, key, expiry_s))


### End-to-end cascade demo

With `invalidate_photo_urls` defined, the cascade from the previous section is
complete. The seed-and-run cell below verifies that one delete call removes the
DB row, six S3 keys (the original + 3 derivatives + 2 face thumbnails), and one
cached presigned URL.


In [ ]:
# Demo: seed DB + cache and run the cascade.
DB = {
    "p-001": PhotoCascadeTargets(
        photo_id="p-001",
        s3_original_key="photos/aa/abcd.jpg",
        s3_original_bucket="my-photos",
        derivative_bucket="my-photo-app-thumbnails",
        derivative_keys={
            "small":  "thumbnails/aa/aaaa/small.jpg",
            "medium": "thumbnails/aa/aaaa/medium.jpg",
            "large":  "thumbnails/aa/aaaa/large.jpg",
        },
        face_thumbnail_keys=["thumbnails/faces/face-001.jpg", "thumbnails/faces/face-002.jpg"],
        summary_id="s-001",
    ),
}
s3 = MagicMock()
cache = _InMemoryRedis()

# Pre-seed the cache for one of the derivative keys so we can verify the invalidation.
await cache.set(_cache_key("my-photo-app-thumbnails", "thumbnails/aa/aaaa/small.jpg", 3600),
                "https://s3/p001/small.jpg", ex=3600)
assert await cache.get(_cache_key("my-photo-app-thumbnails", "thumbnails/aa/aaaa/small.jpg", 3600))

await delete_photo_cascade("p-001", DB, s3, cache)

# Verify S3 deletes (6 expected keys: original + 3 derivatives + 2 face thumbs).
print(f"s3 delete_object call count: {s3.delete_object.call_count}  (expected 6)")
# Verify the row is gone.
print(f"db still has p-001: {'p-001' in DB}  (expected False)")
# Verify cache was invalidated.
remaining = await cache.get(_cache_key("my-photo-app-thumbnails", "thumbnails/aa/aaaa/small.jpg", 3600))
print(f"cache entry remaining: {remaining}  (expected None)")


## Best-Effort vs. Two-Phase

S3 `DeleteObject` is not transactional and is not atomic with the Postgres commit. We pick **best-effort after commit** as the policy: the row goes away first (so no client ever sees it again), S3 deletes run afterward, and a periodic reaper sweeps any `(bucket, key)` pairs reachable via the audit log that are still on S3. The reaper is a single full-bucket `list_objects_v2` filtered against the live `photos` rows; mismatches get deleted.

:::{.callout-note}
The alternative is tombstoned deletion: write a "deletion_request" row before deleting anything, run S3 + Redis cleanup as a separate job, mark the row "completed" only when every side object is gone. This is more robust but adds a new persistence class. For a single-family photo library the best-effort + reaper combo is simpler and loses nothing in practice.

:::


## Soft-Delete Tombstones With a 30-Day Grace Period

Hard deletes are unrecoverable. The user clicks "delete" on the wrong photo on the wrong laptop — they meant `p-099`, the cursor slipped to `p-098` — and the cascade above wipes the only copy of the original on S3. Thirty days later when they notice the missing vacation photo, there is nothing left.

We add a `deleted_at` column on `photos`. `DELETE /photos/{id}` does not physically delete; it sets `deleted_at = now()`. A reaper process runs nightly, deleting every row with `deleted_at < now() - INTERVAL '30 days'`. A row with `deleted_at` set disappears from browse and search (the relevant queries add `WHERE deleted_at IS NULL`), but a `POST /photos/{id}/restore` reverts the soft-delete within the grace period.

:::{.callout-caution}
S3 retains nothing for the same 30 days for free — once `delete_object` is called, S3 bills you only for storage you actually use; objects are gone immediately. The soft-delete grace lives entirely in Postgres. If the user soft-deletes a photo, then 30 days pass, the reaper deletes the row **and** calls the S3 cascade. Only at that point the original is gone.

:::


In [ ]:
from datetime import datetime, timedelta, timezone


SOFT_DELETE_GRACE_DAYS = 30


@dataclass
class SoftPhotosRow:
    """Mirrors the `photos` row with a `deleted_at` column."""
    photo_id:   str
    s3_key:     str
    deleted_at: datetime | None = None


# In-memory table.
PHOTOS_TABLE: dict[str, SoftPhotosRow] = {}


async def soft_delete_photo(photo_id: str, db: dict[str, SoftPhotosRow]) -> bool:
    """Soft-delete: set deleted_at instead of removing the row. Idempotent."""
    row = db.get(photo_id)
    if row is None:
        return False
    if row.deleted_at is None:
        row.deleted_at = datetime.now(timezone.utc)
    return True


async def restore_photo(photo_id: str, db: dict[str, SoftPhotosRow]) -> bool:
    """Restore an accidentally-deleted row. Fails if the grace period expired."""
    row = db.get(photo_id)
    if row is None or row.deleted_at is None:
        return False
    if datetime.now(timezone.utc) - row.deleted_at > timedelta(days=SOFT_DELETE_GRACE_DAYS):
        return False
    row.deleted_at = None
    return True


async def hard_reap(db: dict[str, SoftPhotosRow]) -> int:
    """Process expired tombstones. Returns the count of rows hard-deleted.
    In production this also calls the S3+Redis cascade from the prior section."""
    expired_ids = [pid for pid, row in db.items()
                   if row.deleted_at is not None
                   and datetime.now(timezone.utc) - row.deleted_at > timedelta(days=SOFT_DELETE_GRACE_DAYS)]
    for pid in expired_ids:
        del db[pid]
    return len(expired_ids)


# Demo: soft-delete a photo, try to restore inside grace, attempt to delete an
# already-restored one, advance time, run reaper.
PHOTOS_TABLE["p-100"] = SoftPhotosRow("p-100", "photos/aa/p100.jpg")

removed = await soft_delete_photo("p-100", PHOTOS_TABLE)
print(f"soft-deleted: {removed}  deleted_at={PHOTOS_TABLE['p-100'].deleted_at}")
restored = await restore_photo("p-100", PHOTOS_TABLE)
print(f"restored in grace: {restored}  deleted_at={PHOTOS_TABLE['p-100'].deleted_at}")

# Now delete again, then advance past 30 days via direct manipulation.
await soft_delete_photo("p-100", PHOTOS_TABLE)
PHOTOS_TABLE["p-100"].deleted_at = datetime.now(timezone.utc) - timedelta(days=31)
n_reaped = await hard_reap(PHOTOS_TABLE)
print(f"reaper hard-deleted: {n_reaped}  (expected 1)")
print(f"p-100 still present: {'p-100' in PHOTOS_TABLE}  (expected False)")


## EXIF Extraction and GPS Stripping

EXIF is data the photographer embedded at capture time — camera model, lens, exposure, but also GPS coordinates lost into every picture taken by a phone with location services on. Notebook 06's pipeline extracts a subset of EXIF into `photos` columns (`taken_at`, `gps_lat`, `gps_lng`, `camera_make`, `camera_model`). Necessary for search ("Bondi in July 2023") and for the timeline (sort). But **served to clients** those same fields let anyone who downloads a thumbnail triangulate the photographer's home.

Actually the derivatives we serve contain EXIF bytes as well: Pillow's `Image.save` copies EXIF into the output unless you pass `exif=b""`. So even metadata the API does not include in its JSON response still leaks via the JPEG bytes. The fix: when generating derivatives, strip all metadata from the output bytes. The DB retains a normalized subset (`gps_lat`, `gps_lng`) the API uses for search; the served JPEG contains nothing.


In [ ]:
import io
from typing import BinaryIO


def generate_strip_metadata(img_bytes: bytes, long_edge: int, quality: int) -> bytes:
    """Resize and strip all metadata from the output.

    In Pillow: pass `exif=b""` to `Image.save`. Alternatively open the image
    without EXIF (the `getexif()` API populates a dict but is read-only; re-save
    with an empty exif). The simplest portable recipe is `Image.save(...,
    exif=b"")` or `Image.save(..., exif=None)` on Pillow >= 9 — both produce
    JPEG bytes without an APP1 EXIF segment.

    The full recipe including resize is below.
    """
    try:
        from PIL import Image as PILImage
    except ImportError:
        return img_bytes  # Stub: pass-through without metadata stripping.

    img = PILImage.open(io.BytesIO(img_bytes))
    w, h = img.size
    if max(w, h) > long_edge:
        if w >= h:
            new_w, new_h = long_edge, int(h * long_edge / w)
        else:
            new_w, new_h = int(w * long_edge / h), long_edge
        img = img.resize((new_w, new_h), PILImage.LANCZOS)
    img = img.convert("RGB")            # Force 3-channel → no alpha-as-metadata
    buf = io.BytesIO()
    # exif=b"" is the actual strip. optimize=True makes the file smaller.
    img.save(buf, format="JPEG", quality=quality, exif=b"", optimize=True)
    return buf.getvalue()


# Demo: build a derivative and inspect its bytes for EXIF marker (0xFFE1).
# Without Pillow installed, _resize_lanczos returns original (so no metadata strip
# in this stub), but we can verify the function signature is the production shape.
sample_in = b"\xff\xd8\xff\xe1" + b"\x00\x16EXIF data here" + b"\xff\xd9"
out = generate_strip_metadata(sample_in, 256, 70)
has_exif_out = b"\xff\xe1" in out
print(f"output contains APP1/EXIF marker: {has_exif_out}")


For the original-object case (when the user explicitly asks for the original), we **serve what was stored** — we do not destroy the original at index time. The reasoning: the user owns the photo; their original photographer's EXIF is data they captured intentionally and destroying it at index time would be pretend a privacy scrub that does not match the user's mental model of "my own library." Instead, the **API JSON** responseScrubs `gps_lat`/`gps_lng` by default and only returns them if `?include_location=true` is appended — the API strips metadata from API responses; S3-derivative bytes themselves are clean of metadata at file-write time; the original is preserved with EXIF but only served explicitly.


In [ ]:
from pydantic import BaseModel


class PhotoAPIResponse(BaseModel):
    """The shape returned by `GET /photos/{id}`. GPS is opt-in."""
    photo_id:    str
    taken_at:    datetime
    camera_make: str | None = None
    camera_model: str | None = None
    # GPS is NOT included in the default response. Include explicitly via query.
    gps_lat:     float | None = None
    gps_lng:     float | None = None


def serialize_photo(photo_row: dict, include_location: bool = False) -> PhotoAPIResponse:
    """Strip GPS unless explicitly requested. This is the policy mirror of the
    derivative-level EXIF strip: derivative bytes have no EXIF; API JSON has no
    GPS unless explicitly asked for."""
    return PhotoAPIResponse(
        photo_id=photo_row["photo_id"],
        taken_at=photo_row["taken_at"],
        camera_make=photo_row.get("camera_make"),
        camera_model=photo_row.get("camera_model"),
        gps_lat=photo_row.get("gps_lat") if include_location else None,
        gps_lng=photo_row.get("gps_lng") if include_location else None,
    )


photo_db_row = {
    "photo_id":    "p-001",
    "taken_at":    datetime(2023, 7, 14, 17, 42, 0),
    "camera_make": "Apple",
    "camera_model": "iPhone 14",
    "gps_lat":     -33.89,
    "gps_lng":     151.27,
}

print("Default response (no GPS exposed in JSON):")
print(serialize_photo(photo_db_row).model_dump_json(indent=2))
print("\nWith ?include_location=true (GPS exposed on explicit request):")
print(serialize_photo(photo_db_row, include_location=True).model_dump_json(indent=2))


## Per-Device Audit Log

Every mutating endpoint (upload, delete, restore, pipeline-start, summary-regenerate) writes a row to the `audit_log` table with the responsible `device_id`. The audit log is non-blocking: it is appended in the same transaction as the mutation when possible, else async-fired. For this single-family deployment the audit log exists for *accountability* (who deleted the photo?) and *debugging* (what did the second laptop do?), not for compliance or forensics.


In [ ]:
import enum


class AuditAction(str, enum.Enum):
    upload_photo       = "upload_photo"
    soft_delete_photo  = "soft_delete_photo"
    restore_photo      = "restore_photo"
    pipeline_start     = "pipeline_start"
    pipeline_pause     = "pipeline_pause"
    summary_regenerate = "summary_regenerate"


@dataclass
class AuditEntry:
    audit_id:   str
    device_id:  str
    action:     AuditAction
    target_id:  str
    metadata_json: str
    occurred_at: datetime


AUDIT_LOG: list[AuditEntry] = []


async def record_audit(
    device_id: str,
    action:    AuditAction,
    target_id: str,
    metadata: dict | None = None,
) -> AuditEntry:
    """Append an audit entry. In production this is an INSERT into audit_log
    executed in the same transaction as the mutation whenever possible.
    """
    entry = AuditEntry(
        audit_id=uuid.uuid4().hex,
        device_id=device_id,
        action=action,
        target_id=target_id,
        metadata_json=json.dumps(metadata or {}),
        occurred_at=datetime.now(timezone.utc),
    )
    AUDIT_LOG.append(entry)
    return entry


# Demonstrate: simulate a delete issued from device "device-wife".
await record_audit("device-wife", AuditAction.soft_delete_photo, "p-001",
                   metadata={"reason": "duplicate of p-002"})
await record_audit("device-particle", AuditAction.restore_photo, "p-001",
                   metadata={"reason": "misclick"})

print(f"audit log entries: {len(AUDIT_LOG)}")
for e in AUDIT_LOG:
    print(f"  [{e.occurred_at.isoformat(timespec='seconds')}] {e.device_id}  "
          f"{e.action.value:<22}  target={e.target_id}  meta={e.metadata_json}")


## Per-Owner Export and Purge

Two endpoints complete the family-share story:

- `GET /devices/{device_id}/export` returns every `photos` row whose `owner_device_id = device_id` plus the EXIF/summary/face records the API maintains for those photos (no other device's data). The response is a single JSON or zip.

- `DELETE /devices/{device_id}/purge` runs the delete cascade (above) for every photo owned by that device. This is the GDPR Article 17 analogue for the family scenario: when a family member's laptop leaves the family, their photos leave the library.


In [ ]:
async def export_for_device(device_id: str, photos_by_device: dict[str, list[dict]]) -> list[dict]:
    """Return all photos owned by `device_id`. Plus their summaries + faces.
    In SQL: SELECT * FROM photos WHERE owner_device_id = :did
           JOIN summaries ... JOIN faces ...
    """
    return photos_by_device.get(device_id, [])


async def purge_device(device_id: str,
                       photos_by_device: dict[str, list[dict]],
                       delete_cascade_fn) -> int:
    """Cascade-delete every photo owned by `device_id`.

    delete_cascade_fn is the function from the top of this notebook. Returning
    an int because purging is heavy; the API returns the count."""
    purged = 0
    for photo in photos_by_device.get(device_id, []):
        await delete_cascade_fn(photo["photo_id"], photo, MagicMock(), _InMemoryRedis())
        purged += 1
    photos_by_device[device_id] = []
    await record_audit(device_id, AuditAction.soft_delete_photo,
                       target_id=device_id, metadata={"action": "purge_all"})
    return purged


# Demo: simulate one device with 3 photos, purge, verify the audit log captures it.
photos_by_device: dict[str, list[dict]] = {
    "device-spouse": [
        {"photo_id": "p-100", "s3_key": "photos/x.jpg"},
        {"photo_id": "p-101", "s3_key": "photos/y.jpg"},
        {"photo_id": "p-102", "s3_key": "photos/z.jpg"},
    ],
}
export_before = await export_for_device("device-spouse", photos_by_device)
print(f"exportable before purge: {len(export_before)} photos")

# The purge call below uses the simplified inline cascade (no S3 mocks — we just
# drop them from the map). Production uses delete_photo_cascade.
async def fake_cascade(pid, photo, s3, cache):
    pass

purged = await purge_device("device-spouse", photos_by_device, fake_cascade)
export_after = await export_for_device("device-spouse", photos_by_device)
print(f"purged: {purged} photos")
print(f"exportable after purge: {len(export_after)} photos")
last_audit = AUDIT_LOG[-1]
print(f"audit: {last_audit.device_id} {last_audit.action.value} meta={last_audit.metadata_json}")


## Schema Additions

Three new columns on `photos`, one new `audit_log` table, plus the partial index for live rows.

```sql
-- 0022_pht_lifecycle.sql
ALTER TABLE photos
    ADD COLUMN IF NOT EXISTS deleted_at       TIMESTAMPTZ,
    ADD COLUMN IF NOT EXISTS gps_lat          DOUBLE PRECISION,
    ADD COLUMN IF NOT EXISTS gps_lng          DOUBLE PRECISION;

CREATE INDEX ix_photos_deleted_at ON photos (deleted_at)
    WHERE deleted_at IS NOT NULL;       -- reaper query is one seek

CREATE TABLE audit_log (
    audit_id      TEXT PRIMARY KEY,
    device_id     TEXT NOT NULL REFERENCES devices(device_id),
    action        TEXT NOT NULL,           -- one of upload_photo | soft_delete_image | ...
    target_id     TEXT NOT NULL,           -- photo_id/device_id/job_id
    metadata_json JSONB NOT NULL DEFAULT '{}'::jsonb,
    occurred_at   TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX ix_audit_log_device ON audit_log (device_id, occurred_at DESC);
CREATE INDEX ix_audit_log_target ON audit_log (target_id, occurred_at DESC);
```

:::{.callout-note}
Every query that previously selected uncritically from `photos` MUST add `WHERE deleted_at IS NULL`. Forgetting this in one of N queries (summary-view, search, timeline) leaks soft-deleted photos. Application-enforced soft delete via a SQLAlchemy event listener that injects `WHERE deleted_at IS NULL` into every select on `photos` is a robust alternative — pay for one global filter, ship without auditing each query site.

:::


## Summary

This notebook gave the photo app its data-lifecycle spine. We built a cascading delete that wipes Postgres rows, S3 originals, derivative objects, face-thumbnail objects, and Redis cache entries in the right order with the right transaction boundaries. A 30-day soft-delete grace period recovers misclicks; a nightly reaper drains the queue. EXIF is extracted to DB columns for search but **stripped from every derivative byte** and **scrubbed from API responses unless explicitly requested**. Per-`device_id` audit logging records every mutation. Per-`device_id` export and purge close the "this laptop leaves the family" case.

**What changed relative to notebook 12.** New `DELETE /photos/{id}`, `POST /photos/{id}/restore`, `GET /devices/{id}/export`, `DELETE /devices/{id}/purge` endpoints. Three new columns on `photos` (`deleted_at`, `gps_lat`, `gps_lng`). One new `audit_log` table. The derivative generator from PHT:02 now strips metadata at file-write time (`exif=b""`).

**What this enables for the rest of PHT.** PHT:07's metrics now include "deletes per day", "restores per day", and "purge size per device" alongside the operational metrics. The audit log becomes the source of truth for the `/changes` endpoint that the Flet "Recent Activity" view that we cut from PHT:01 would eventually want.

---



---


■
